# Student Placement Prediction using a Decision Tree

In this notebook we build a complete machine learning project step by step.
We use simple code and explain every step in simple English.

In [4]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'numpy'

---
## Step 1: Problem

**Real-life story:** A college placement cell wants to know which students are likely to get a job placement.
If they know this early, they can give extra help (mock interviews, practice tests, mentoring) to the students who need it.

**What we are predicting:** `placement_status` (Placed or Not Placed)

**Type of problem:** Classification, because the answer is one of two classes.

**What we use to predict (features):**
- Student details: gender, branch, extracurricular activities
- Habits: study hours, sleep hours, internet usage
- Performance: attendance, assignments completed, previous score, exam score

**Two types of mistakes the model can make**
- It says "Placed" but the student is not placed: the student may miss the extra help.
- It says "Not Placed" but the student is placed: the college wastes some effort.

---
## Step 2: Data + EDA (Find patterns)

EDA means **Exploratory Data Analysis**.
It is like meeting a new student for the first time: first you look, ask questions and understand, then you decide what to do.

### 2.1 Load the data

In [ ]:
from pathlib import Path

project_dir = Path.cwd()
candidate_paths = [
    project_dir,
    project_dir.parent,
    project_dir / 'Decision_Tree',
]
data_path = None
for base in candidate_paths:
    candidate = base / 'student_performance_dataset.csv'
    if candidate.exists():
        data_path = candidate
        break
if data_path is None:
    raise FileNotFoundError("student_performance_dataset.csv was not found in the project folder.")

df_copy = pd.read_csv(data_path)
# df_copy = df.copy()
# df.head(5)
df_copy.head(5)


### 2.2 Understand the columns

| Column | Meaning |
|---|---|
| `student_id` | Unique ID of the student (only a label, not useful for prediction) |
| `gender` | Male or Female |
| `branch` | CSE, IT, ECE, Mechanical, Civil, Commerce |
| `study_hours` | Study hours per day |
| `attendance` | Attendance in percent |
| `sleep_hours` | Sleep hours per day |
| `internet_usage` | Internet use in hours per day |
| `assignments_completed` | Number of assignments completed |
| `previous_score` | Marks in the previous exam |
| `extracurricular` | Does the student take part in extra activities? |
| `exam_score` | Latest exam marks |
| `placement_status` | **Target**: Placed or Not Placed |

In [ ]:
# Data types and non-null counts
df_copy.info()

In [ ]:
# Quick summary of the number columns
df_copy.describe()

In [ ]:
# How many missing values are in each column?
df_copy.isnull().sum()

In [ ]:
# Are there duplicate rows?
df_copy.duplicated().sum()

In [ ]:
# check Columns purity
cat_col = df_copy.select_dtypes(include=['object', 'string']).columns
cat_col
print(cat_col)
for col in cat_col:
  print(df_copy[col].unique())
  print()


### What is wrong with this data?

Real-world data is never perfect. Here we can see:

1. **Missing values** in many columns (study hours, attendance, sleep hours, and more).
2. **Duplicate rows**, the same student appears twice.
3. **Gender is written in many ways**: `Male`, `male`, `M`, `Female`, `female`, `F`.
4. **Branch has spelling problems**: `CSE`, `cse`, ` CSE ` (with extra spaces), `MECHANICAL`, `mechanical`.
5. **Impossible values (outliers)**: check `study_hours`, `sleep_hours` and `internet_usage` in the table above.
   Nobody can study 21 hours every day!

We will fix these in Step 3 and Step 4.

### 2.3 Is our target balanced?

In [ ]:
sns.countplot(df_copy['placement_status'])

#

### 2.4 Do placed students look different?
Let us compare the average of every number column for the two groups.

In [ ]:
# Group by num_col to placement_status
num_col = df_copy.select_dtypes(include =np.number).columns
df_copy.groupby('placement_status')[num_col].mean()

In [ ]:
# Box plots: compare Placed and Not Placed for each number column
num_cols = df_copy.select_dtypes(include=np.number).columns

for col in num_cols:
    sns.boxplot(x="placement_status", y=col, data=df_copy)
    plt.title(col)
    plt.show()

In [ ]:
# Correlation: which number columns move together?
sns.heatmap(df_copy[num_col].corr(), annot = True)

---
## Step 3: Clean data

Rule: **never change the original data**. We work on a copy, so we can always go back.

We will do 5 cleaning jobs:
1. Remove duplicate rows
2. Drop `student_id` (it is only a label)
3. Fix the `gender` column
4. Fix the `branch` column
5. Fix impossible values (outliers)



In [ ]:
# Make a copy so the original df stays safe

# Job 1: remove duplicate rows
df_copy.drop_duplicates(inplace = True)
df_copy.duplicated().sum()
# Job 1.0: Remove null values
df_copy.dropna(inplace=True)
df_copy.isnull().sum()

In [ ]:
# Job 2: student_id is only a label. It does not help to predict placement.
df_copy.drop(columns=['student_id'], inplace=True)
df_copy.head(2)

In [ ]:
# Job 3: fix gender
# strip() removes extra spaces, lower() makes small letters, map() gives one clean name
df_copy['gender'] = df_copy['gender'].str.strip().str.lower().map({'male': 'male', 'female': 'female', 'm': 'male', 'f': 'female'})
df_copy['gender'].unique()

In [ ]:
# Job 4: fix branch
# Remove extra spaces and make everything CAPITAL letters, so ' cse ' becomes 'CSE'
df_copy['branch'] = df_copy['branch'].str.strip().str.upper()
df_copy['branch'].unique()

In [ ]:
# Job 5: fix impossible values with clip()
# clip(0, 12) means: anything above 12 becomes 12, anything below 0 becomes 0
df_copy["study_hours"] = df_copy["study_hours"].clip(0, 12)
df_copy["sleep_hours"] = df_copy["sleep_hours"].clip(3, 12)
df_copy["internet_usage"] = df_copy["internet_usage"].clip(0, 12)

df_copy[["study_hours", "sleep_hours", "internet_usage"]].describe().round(2)




In [ ]:
# Quick look at the cleaned categories
# Box plots: compare Placed and Not Placed for each number column
num_col = ['study_hours', 'sleep_hours', 'internet_usage']

for col in num_col:
    sns.boxplot(x="placement_status", y=col, data=df_copy)
    plt.title(col)
    plt.show()

Cleaning is done. Values are now consistent and there are no impossible numbers.
The missing values are still there. We fix them next.

---
## Step 4: Preprocessing

A decision tree needs clean numbers. Preprocessing has 4 small jobs:

1. **Separate X and y.** `X` = the inputs, `y` = the answer we want to predict.
2. **Split into train and test (80% / 20%).**
   *Real-life example:* Students learn from practice papers (train data) and the final exam (test data) has questions they have never seen. The test data tells us the truth.
3. **Convert text to numbers (one-hot encoding).**
   The column `branch` becomes separate columns like `branch_CSE = 1 or 0`, `branch_IT = 1 or 0`, and so on.

**Do we need scaling?** No. Decision trees ask questions like "attendance above 75?", so the scale of the numbers does not matter.

In [ ]:
# 1. Convert text to numbers (one-hot encoding) on branch and others label encoding.
from sklearn.preprocessing import LabelEncoder

# Create a copy
df_copy = df_copy.copy()

# Branch → One-Hot Encoding
df_copy = pd.get_dummies(
    df_copy,
    columns=["branch"],
    dtype=int
)

# Label Encoding
label_en = LabelEncoder()

df_copy["gender"] = label_en.fit_transform(df_copy["gender"])
df_copy["extracurricular"] = label_en.fit_transform(df_copy["extracurricular"])
df_copy["placement_status"] = label_en.fit_transform(df_copy["placement_status"])

df_copy.head()

In [ ]:
# 2. Separate X and y.
X = df_copy.drop(columns = ['placement_status'])
y = df_copy['placement_status']


In [ ]:
# 3. Split into train and test (80% / 20%).
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

---
## Step 5: Model
**Model : Default decision tree.**


In [ ]:
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)

compare = pd.DataFrame({
    'Actul' : y_test,
    'Predicted' : y_pred
})
compare.head(10)


In [ ]:
from sklearn.metrics import accuracy_score
Test_acc = accuracy_score(y_test, clf.predict(X_test))
Train_acc = accuracy_score(y_train, clf.predict(X_train))
print(f'Train accuracy: {Train_acc}')
print(f'Test accuracy: {Test_acc}')

---
## Step 6: Tune

Tuning means finding the best **settings** (hyperparameters) for the tree.

| Setting | Meaning | Real-life idea |
|---|---|---|
| `max_depth` | Maximum number of question levels | Limit an interview to 3 rounds |
| `min_samples_leaf` | Minimum students allowed in a final group | Do not make a rule from just 2 students |
| `criterion` | How to measure "messy" groups: `gini` or `entropy` | Two different rulers for the same job |

We train trees with depth 1, 2, 3 ... 15 and plot train vs test accuracy.

In [ ]:
# 4. Define hyperparameters
param_grid = {
    'max_depth':[3,5,7,10],
    'min_samples_leaf':[1,2,5,10],
    'criterion':['gini','entropy']
}

In [ ]:
# RandomizedSearch / gridSearchCV
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator = clf,
    param_distributions = param_grid,
    n_iter = 10,
    cv = 5,
    scoring = 'accuracy',
    random_state = 42
)

In [ ]:
# Train
random_search.fit(X_train, y_train)

In [ ]:
# Best parameters
print(random_search.best_params_)

In [ ]:
# Best Model
best_model = random_search.best_estimator_

In [ ]:
# prediction
y_pred = best_model.predict(X_test)

In [ ]:
# final accuracy score
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test, y_pred))

In [ ]:
# Calculate and print train/test accuracy for the tuned (best) model
tuned_train_acc = accuracy_score(y_train, best_model.predict(X_train))
tuned_test_acc = accuracy_score(y_test, best_model.predict(X_test))

print(f'Tuned Model - Train Accuracy: {tuned_train_acc:.4f}')
print(f'Tuned Model - Test Accuracy: {tuned_test_acc:.4f}')

---
## Step 7: Evaluate

Now we test the tuned model on the **test data**, which it has never seen.

**Metrics in simple words** (think of "Placed" as the positive class):

| Metric | Question it answers |
|---|---|
| Accuracy | Out of all students, how many did we predict correctly? |
| Precision | Out of students we said "Placed", how many were really placed? |
| Recall | Out of students who were really placed, how many did we find? |
| F1-score | One number that balances precision and recall |


**Confusion matrix** is a 2x2 table that shows correct and wrong predictions of each type.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
print(f'Accuracy : {accuracy_score(y_test, y_pred)  }')
print(f'Precision : {precision_score(y_test, y_pred)  }')
print(f'Recall : {recall_score(y_test, y_pred)  }')
print(f'F1-score : {f1_score(y_test, y_pred)  }')
print(f'Confusion matrix : \n{confusion_matrix(y_test, y_pred)  }')

---
## Step 8: Explain

A big reason to use a decision tree is that we can **explain** its decisions.
A placement officer will not trust a model that only says "Placed" without a reason.

### Feature importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get feature importances from our best model
importances = best_model.feature_importances_
features = X.columns

# Create a DataFrame for easy sorting and plotting
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot the top 10 most important features
plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance_df.head(10),
    palette='viridis'
)
plt.title('Top 10 Most Important Features (Tuned Decision Tree)')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Step 9: Save model


In [ ]:
import joblib

# Save the trained best model to a file
model_filename = 'placement_model.joblib'
joblib.dump(best_model, model_filename)

print(f"Model successfully saved as '{model_filename}'!")

In [ ]:
import joblib
joblib.dump(X.columns.tolist(), 'columns.joblib')
print('columns.joblib saved!')


**Next: build the FastAPI app**
- `POST /predict` takes student details and returns Placed / Not Placed with the probability.
- A simple web page (HTML form) calls this API and shows the result.
- Keep `placement_model.joblib` and the same scikit-learn version in the API project.

#